In [ ]:
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
# ================================
# 1. Setup & Hyperparameters
# ================================

BASE_SEED = 44

torch.manual_seed(BASE_SEED)
np.random.seed(BASE_SEED)
random.seed(BASE_SEED)

# Device selection
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")

# Enable fast matmul on modern NVIDIA GPUs
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

# ---- Experiment hyperparameters ----
N = 15  # number of completions per run (for training)
K = 15  # number of preference pairs to sample per run
epochs = 50  # DPO training epochs
NUM_RUNS = 10  # number of runs / topics

# Number of completions for evaluation per model at end of each run
EVAL_N = 15

# ---- Generation hyperparameters ----
MAX_NEW_TOKENS = 256
GEN_BATCH_SIZE = 15
GEN_TOP_K = 50

# ---- Bradley–Terry hyperparameters ----
BT_N_ITERS = 1000
BT_LR = 0.01
BT_L1_REG = 0.01

# ---- DPO hyperparameters ----
DPO_LR = 1e-5
DPO_BETA = 0.1

# ---- Model names (replace with actual HF IDs you use) ----
policy_model_name = "Qwen/Qwen3-1.7B"
reward_model_name = "Skywork/Skywork-Reward-V2-Qwen3-1.7B"

# ================================
# 2. Tokenizer & Reward Model
# ================================

# Shared tokenizer for policy + reward model (since RM is Qwen3-based)
tokenizer = AutoTokenizer.from_pretrained(policy_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Reward model is fixed across runs
reward_model = AutoModelForSequenceClassification.from_pretrained(reward_model_name)
reward_model.to(device)
reward_model.eval()

# ================================
# 3. Topics / Prompts for Runs
# ================================

topics = [
    "democracy",
    "technology",
    "education",
    "climate",
    "healthcare",
    "economics",
    "ethics",
    "artificial intelligence",
    "privacy",
    "globalization",
]  # 10 topics, one per run (run_idx 0..9)

assert len(topics) >= NUM_RUNS, "Need at least NUM_RUNS topics."

# ================================
# 4. Helper Functions
# ================================


def generate_essay_completions(
    model,
    tokenizer,
    prompt,
    n=100,
    max_new_tokens=MAX_NEW_TOKENS,
    run_idx=0,
):
    """
    Generate n completions for an essay prompt using the policy model.
    """
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    completions = []

    batch_size = GEN_BATCH_SIZE
    # We may generate a bit more than n if n is not divisible by batch_size
    num_batches = n // batch_size if n % batch_size == 0 else n

    with torch.no_grad():
        for _ in tqdm(
            range(num_batches),
            desc=f"Run {run_idx + 1}: Generating completions (Qwen3)",
            leave=False,
        ):
            # For simplicity we still generate GEN_BATCH_SIZE at a time;
            # if n is not divisible by GEN_BATCH_SIZE, we may slightly overshoot.
            batch_inputs = {k: v.repeat(batch_size, 1) for k, v in inputs.items()}
            outputs = model.generate(
                **batch_inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_k=GEN_TOP_K,
                pad_token_id=tokenizer.pad_token_id,
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            # Remove the prompt prefix to keep only the completion text if desired
            stripped = [
                d[len(prompt) :] if d.startswith(prompt) else d for d in decoded
            ]
            completions.extend(stripped)

            if len(completions) >= n:
                completions = completions[:n]
                break

    return completions


def get_reward_scores(
    reward_model, tokenizer, prompt, completions, max_length=512, batch_size=4
):
    """
    Use the reward model to assign a scalar score to each completion.
    Higher score = better according to reward model.
    """
    reward_model.eval()
    scores = []

    for i in range(0, len(completions), batch_size):
        batch_comps = completions[i : i + batch_size]
        texts = [f"{prompt}\n\n{comp}" for comp in batch_comps]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)

        with torch.no_grad():
            outputs = reward_model(**inputs)
            # Assume scalar reward in logits[..., 0]
            batch_scores = outputs.logits.squeeze(-1).detach().cpu().numpy()
            scores.extend(batch_scores.tolist())

    return np.array(scores)


# ---- Preference Oracle & Graph Estimation (Bradley–Terry) ----


def sample_pairs(n_completions, n_pairs):
    """
    Uniformly sample unique unordered pairs (i, j) with i < j.
    """
    pairs = []
    existing = set()
    while len(pairs) < n_pairs:
        idx = np.random.choice(n_completions, 2, replace=False)
        idx = tuple(sorted(idx))
        if idx not in existing:
            existing.add(idx)
            pairs.append(idx)
    return pairs


def label_pairs(pairs, scores):
    """
    Given a list of pairs (i, j) and scores for each completion,
    label them so that the higher-scoring index is the winner.
    Returns list of (winner, loser) indices.
    """
    labeled_pairs = []
    for i, j in pairs:
        if scores[i] > scores[j]:
            labeled_pairs.append((i, j))  # i wins
        else:
            labeled_pairs.append((j, i))  # j wins
    return labeled_pairs


def fit_bradley_terry(
    n_completions,
    labeled_pairs,
    n_iters=BT_N_ITERS,
    lr=BT_LR,
    l1_reg=BT_L1_REG,
):
    """
    Fit a Bradley-Terry model to estimate per-completion scores
    from a subset of labeled preference pairs.
    """
    scores = torch.randn(n_completions, requires_grad=True)
    optimizer = torch.optim.Adam([scores], lr=lr)

    winners = torch.tensor([w for w, _ in labeled_pairs], dtype=torch.long)
    losers = torch.tensor([l for _, l in labeled_pairs], dtype=torch.long)

    for _ in range(n_iters):
        optimizer.zero_grad()
        diffs = scores[winners] - scores[losers]
        # Bradley–Terry likelihood (negative log-likelihood)
        loss = -torch.sum(torch.log(torch.sigmoid(diffs) + 1e-8))
        # Small L1 regularization for stability
        loss += l1_reg * torch.sum(torch.abs(scores))
        loss.backward()
        optimizer.step()

    scores_np = scores.detach().cpu().numpy()
    # Normalize scores for interpretability
    scores_np = (scores_np - scores_np.mean()) / (scores_np.std() + 1e-8)
    return scores_np


def construct_full_graph(scores):
    """
    Given scores for each completion (e.g., from Bradley–Terry),
    construct the full preference graph over all (i, j),
    returning list of (winner, loser) pairs.
    """
    n = len(scores)
    pairs = []
    for i in range(n - 1):
        for j in range(i + 1, n):
            if scores[i] > scores[j]:
                pairs.append((i, j))  # i wins
            else:
                pairs.append((j, i))  # j wins
    return pairs


# ---- Fast batched log-probs ----


def get_log_prob_sums_from_inputs(model, input_ids, attention_mask):
    """
    Compute sum of log p(tokens) per sequence for a batch.

    input_ids: [B, L]
    attention_mask: [B, L]
    Returns: tensor [B]
    """
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]  # [B, L-1, V]
    labels = input_ids[:, 1:]  # [B, L-1]

    log_probs = nn.functional.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(dim=-1, index=labels.unsqueeze(-1)).squeeze(
        -1
    )  # [B, L-1]

    mask = attention_mask[:, 1:]  # [B, L-1]
    sum_log_probs = (token_log_probs * mask).sum(dim=-1)  # [B]
    return sum_log_probs


# ---- Fast DPO Training Loop ----


def train_dpo_epoch_save_log_probs_fast(
    policy_model,
    ref_log_probs_tensor,  # [N], on device
    input_ids_all,  # [N, L], on device
    attention_mask_all,  # [N, L], on device
    pairs,  # list of (winner_idx, loser_idx)
    epochs=10,
    lr=DPO_LR,
    beta=DPO_BETA,
    run_idx=0,
    desc_prefix="",
):
    """
    Much faster DPO training:
    - One forward over ALL completions per epoch
    - No tokenization or ref model forward in the loop
    """
    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=lr)

    winners = torch.tensor([w for w, _ in pairs], dtype=torch.long, device=device)
    losers = torch.tensor([l for _, l in pairs], dtype=torch.long, device=device)

    log_probs_over_time = []

    use_amp = device == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    for epoch in range(epochs):
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            # Forward once on all completions
            policy_log_probs = get_log_prob_sums_from_inputs(
                policy_model, input_ids_all, attention_mask_all
            )  # [N]

            # Gather for the pairs
            policy_w = policy_log_probs[winners]
            policy_l = policy_log_probs[losers]

            ref_w = ref_log_probs_tensor[winners]
            ref_l = ref_log_probs_tensor[losers]

            logits = beta * ((policy_w - ref_w) - (policy_l - ref_l))
            losses = -nn.functional.logsigmoid(logits)
            loss = losses.mean()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Save per-example log-probs for metrics (just detach)
        log_probs_over_time.append(policy_log_probs.detach().cpu().numpy().tolist())

        if (epoch + 1) % 50 == 0:
            print(
                f"{desc_prefix} Run {run_idx + 1}: "
                f"Epoch {epoch + 1}/{epochs}, loss={loss.item():.4f}"
            )

    return {"log_probs": log_probs_over_time}


# ---- Agreement metric ----


def count_pair_agreements(model_scores, oracle_scores):
    """
    Count how many pairwise preferences (i,j) the model agrees on
    with the oracle ordering given by oracle_scores.

    model_scores: array-like of shape [N]
    oracle_scores: array-like of shape [N] (oracle ranking scores)
    """
    model_scores = np.array(model_scores)
    oracle_scores = np.array(oracle_scores)
    n = len(oracle_scores)
    assert len(model_scores) == n

    agreements = 0
    total = n * (n - 1) // 2

    for i in range(n):
        for j in range(i + 1, n):
            oracle_pref = oracle_scores[i] - oracle_scores[j]
            model_pref = model_scores[i] - model_scores[j]

            # Agreement if both prefer the same direction
            if oracle_pref * model_pref > 0:
                agreements += 1

    return agreements, total


# ================================
# 5. Multi-run Experiment 2
# ================================

all_ref_frac = []
all_baseline_frac_over_time = []
all_graph_frac_over_time = []

# New: store end-of-run average rewards for each trained model
all_baseline_end_rewards = []
all_graph_end_rewards = []

for run_idx in range(NUM_RUNS):
    print(
        f"\n================ Experiment 2 — Run {run_idx + 1}/{NUM_RUNS} ================"
    )

    # Different seed per run (same scheme as above)
    seed = BASE_SEED + run_idx
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    topic = topics[run_idx]
    prompt = f"Write a detailed, well-structured essay about {topic}."

    print(f"Run {run_idx + 1}: Topic = '{topic}'")

    # 1) Initialize policy models for this run
    pi_ref = AutoModelForCausalLM.from_pretrained(policy_model_name)
    pi_ref.config.use_cache = False
    pi_ref.to(device)
    pi_ref.eval()

    pi_train_baseline = copy.deepcopy(pi_ref)
    pi_train_graph = copy.deepcopy(pi_ref)
    pi_train_baseline.train()
    pi_train_graph.train()

    print("Models initialized: pi_ref, pi_train_baseline, pi_train_graph")

    # 2) Data: completions for this run from the reference policy
    completions = generate_essay_completions(
        pi_ref,
        tokenizer,
        prompt,
        n=N,
        max_new_tokens=MAX_NEW_TOKENS,
        run_idx=run_idx,
    )
    print(f"Run {run_idx + 1}: Generated {len(completions)} completions.")

    # 3) Oracle ranking via reward model scores
    reward_scores = get_reward_scores(
        reward_model,
        tokenizer,
        prompt,
        completions,
    )  # shape [N]

    oracle_scores = reward_scores  # higher = better according to RM

    print(f"Run {run_idx + 1}: Reward model scores (oracle):")
    for i, s in enumerate(oracle_scores):
        print(f"  Example {i}: score {s:.4f}")

    # 4) Sample K oracle comparisons ("human feedback" simulated via reward model)
    sampled_indices = sample_pairs(N, K)
    labeled_sample = label_pairs(sampled_indices, oracle_scores)

    # 5) Estimate scores with Bradley–Terry from the sampled pairs
    estimated_scores = fit_bradley_terry(N, labeled_sample)

    # 6) Construct full graph from estimated BT scores
    bt_full_graph_pairs = construct_full_graph(estimated_scores)

    print(f"Run {run_idx + 1}: Sampled {K} pairs.")
    print(
        f"Run {run_idx + 1}: Constructed BT-estimated full graph with "
        f"{len(bt_full_graph_pairs)} pairs."
    )
    print(
        f"Run {run_idx + 1}: Correlation between RM scores and BT-estimated scores: "
        f"{np.corrcoef(oracle_scores, estimated_scores)[0, 1]:.4f}"
    )

    # 7) Pre-tokenize completions once and compute reference log-probs once
    tokens = tokenizer(completions, return_tensors="pt", padding=True, truncation=True)
    input_ids_all = tokens["input_ids"].to(device)
    attention_mask_all = tokens["attention_mask"].to(device)

    with torch.no_grad():
        ref_log_probs_tensor = get_log_prob_sums_from_inputs(
            pi_ref, input_ids_all, attention_mask_all
        )  # [N]

    ref_log_probs = ref_log_probs_tensor.detach().cpu().numpy().tolist()
    ref_log_probs_tensor = ref_log_probs_tensor.to(device)

    # 8) Train Baseline DPO (on sampled pairs only)
    print(f"Run {run_idx + 1}: Training Baseline DPO (sampled pairs)...")
    metrics_baseline = train_dpo_epoch_save_log_probs_fast(
        pi_train_baseline,
        ref_log_probs_tensor,
        input_ids_all,
        attention_mask_all,
        labeled_sample,  # K RM-labeled pairs
        epochs=epochs,
        lr=DPO_LR,
        beta=DPO_BETA,
        run_idx=run_idx,
        desc_prefix="Baseline",
    )

    # 9) Train Graph DPO (on BT-estimated full graph)
    print(f"Run {run_idx + 1}: Training Graph DPO (BT-estimated full graph)...")
    metrics_graph = train_dpo_epoch_save_log_probs_fast(
        pi_train_graph,
        ref_log_probs_tensor,
        input_ids_all,
        attention_mask_all,
        bt_full_graph_pairs,  # full graph induced by BT scores
        epochs=epochs,
        lr=DPO_LR,
        beta=DPO_BETA,
        run_idx=run_idx,
        desc_prefix="Graph",
    )

    # 10) Pairwise agreement metrics w.r.t. oracle (reward model) ranking
    epochs_trained = len(metrics_baseline["log_probs"])

    ref_agree, total_pairs = count_pair_agreements(ref_log_probs, oracle_scores)
    ref_frac = ref_agree / total_pairs

    baseline_frac_over_time = []
    graph_frac_over_time = []

    for e in range(epochs_trained):
        baseline_scores_e = np.array(metrics_baseline["log_probs"][e])
        graph_scores_e = np.array(metrics_graph["log_probs"][e])

        baseline_agree_e, _ = count_pair_agreements(baseline_scores_e, oracle_scores)
        graph_agree_e, _ = count_pair_agreements(graph_scores_e, oracle_scores)

        baseline_frac_over_time.append(baseline_agree_e / total_pairs)
        graph_frac_over_time.append(graph_agree_e / total_pairs)

    print(f"Run {run_idx + 1}: Total possible pairs: {total_pairs}")
    print(
        f"Run {run_idx + 1}: pi_ref agreement (constant): {ref_agree} / {total_pairs} "
        f"({ref_frac:.3f} fraction)"
    )

    # Final preference alignment percentages for the two trained models
    baseline_final_alignment = baseline_frac_over_time[-1]
    graph_final_alignment = graph_frac_over_time[-1]

    print(
        f"Run {run_idx + 1}: Final preference alignment with reward model "
        f"(pairwise agreement):"
    )
    print(f"  Baseline DPO: {baseline_final_alignment * 100:.2f}%")
    print(f"  Graph DPO:    {graph_final_alignment * 100:.2f}%")

    # 11) Evaluate trained models: sample EVAL_N completions and compute average reward
    print(
        f"Run {run_idx + 1}: Evaluating trained policies with {EVAL_N} new completions each..."
    )

    # Baseline DPO model
    baseline_eval_completions = generate_essay_completions(
        pi_train_baseline,
        tokenizer,
        prompt,
        n=EVAL_N,
        max_new_tokens=MAX_NEW_TOKENS,
        run_idx=run_idx,
    )
    baseline_eval_scores = get_reward_scores(
        reward_model,
        tokenizer,
        prompt,
        baseline_eval_completions,
    )
    baseline_avg_reward = float(np.mean(baseline_eval_scores))

    # Graph DPO model
    graph_eval_completions = generate_essay_completions(
        pi_train_graph,
        tokenizer,
        prompt,
        n=EVAL_N,
        max_new_tokens=MAX_NEW_TOKENS,
        run_idx=run_idx,
    )
    graph_eval_scores = get_reward_scores(
        reward_model,
        tokenizer,
        prompt,
        graph_eval_completions,
    )
    graph_avg_reward = float(np.mean(graph_eval_scores))

    print(f"Run {run_idx + 1}: Average reward over {EVAL_N} new completions:")
    print(f"  Baseline DPO: {baseline_avg_reward:.4f}")
    print(f"  Graph DPO:    {graph_avg_reward:.4f}")

    # Store per-run results
    all_ref_frac.append(ref_frac)
    all_baseline_frac_over_time.append(baseline_frac_over_time)
    all_graph_frac_over_time.append(graph_frac_over_time)

    all_baseline_end_rewards.append(baseline_avg_reward)
    all_graph_end_rewards.append(graph_avg_reward)

    # Clear models to avoid memory issues
    del pi_ref, pi_train_baseline, pi_train_graph
    if device == "cuda":
        torch.cuda.empty_cache()

# ================================
# 6. Aggregate results over runs
# ================================

all_ref_frac = np.array(all_ref_frac)  # shape [NUM_RUNS]
all_baseline_frac_over_time = np.array(
    all_baseline_frac_over_time
)  # [NUM_RUNS, epochs]
all_graph_frac_over_time = np.array(all_graph_frac_over_time)  # [NUM_RUNS, epochs]

avg_ref_frac = all_ref_frac.mean()
avg_baseline_frac_over_time = all_baseline_frac_over_time.mean(axis=0)
avg_graph_frac_over_time = all_graph_frac_over_time.mean(axis=0)

# Average end-of-run rewards for the two trained models
all_baseline_end_rewards = np.array(all_baseline_end_rewards)
all_graph_end_rewards = np.array(all_graph_end_rewards)

avg_baseline_end_reward = all_baseline_end_rewards.mean()
avg_graph_end_reward = all_graph_end_rewards.mean()

print("\n========== EXPERIMENT 2 — AVERAGED RESULTS OVER RUNS ==========")
print(f"Average pi_ref agreement fraction over {NUM_RUNS} runs: {avg_ref_frac:.3f}")
print(
    f"Average final reward over {NUM_RUNS} runs "
    f"(measured on {EVAL_N} new completions per run):"
)
print(f"  Baseline DPO: {avg_baseline_end_reward:.4f}")
print(f"  Graph DPO:    {avg_graph_end_reward:.4f}")

# ================================
# 7. Visualization (Averaged)
# ================================

plt.figure(figsize=(8, 5))
x = list(range(epochs))

plt.plot(
    x,
    [avg_ref_frac] * epochs,
    label="Ref",
    linestyle="--",
)

plt.plot(
    x,
    avg_baseline_frac_over_time,
    label="DPO",
    marker="o",
    markersize=4,
)
plt.plot(
    x,
    avg_graph_frac_over_time,
    label="GGDPO",
    marker="x",
    markersize=4,
)

plt.xlabel("Epochs")
plt.ylabel("Fraction of pairwise agreements w.r.t. reward-model oracle")
plt.title(f"Pairwise Agreement with Reward Model (Averaged over {NUM_RUNS} runs)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
